# COMP64702 RAG Coursework
## Notebook 3: Evaluation

This notebook evaluates generated answers using ROUGE, BLEU,
BERTScore and faithfulness metrics.

### Changes applied from improved version (`03_evaluation_improved.ipynb`)

| Metric / Feature | Original | Improved |
|---|---|---|
| ROUGE-1/2/L | ✅ | ✅ |
| BLEU | ✅ | ✅ |
| BERTScore | ✅ | ✅ |
| Faithfulness | ✅ (token overlap) | ✅ (kept) |
| **Retrieval Precision@1 / @5** | ❌ Missing | ✅ Added |
| **Per-difficulty / type breakdown** | ❌ Missing | ✅ Added |
| **Baseline comparison table** | Prose only | ✅ Numerical delta table |
| **Retrieval audit** | ❌ Missing | ✅ Added |
| **`retrieved_doc_ids` extracted** | ❌ | ✅ (feeds retrieval metrics) |
| **Results as DataFrame** | List of dicts | ✅ `pd.DataFrame` for easy groupby |
| **Baseline delta saved to JSON** | ❌ | ✅ |

In [ ]:
# Install dependencies
# CHANGE: added -q flag for cleaner output
!pip install rouge-score bert-score nltk -q

In [ ]:
# Imports
# CHANGE: added pandas (for DataFrame-based groupby breakdowns);
#         nltk.download now uses quiet=True to suppress noise;
#         added print("Imports OK") for confirmation.

import json
from pathlib import Path
import numpy as np
import pandas as pd

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bertscore_score
import nltk
nltk.download("punkt", quiet=True)

print("Imports OK")

In [ ]:
# Paths and data loading (unchanged)

DATA_DIR = Path(".")

benchmark_path = DATA_DIR / "benchmark_dataset.json"
outputs_path   = DATA_DIR / "test_outputs.json"
eval_path      = DATA_DIR / "evaluation_results.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

with open(outputs_path, "r", encoding="utf-8") as f:
    outputs_payload = json.load(f)

outputs = outputs_payload["results"]
print(f"Loaded {len(benchmark)} benchmark items and {len(outputs)} outputs")

In [ ]:
# Match benchmark and outputs by query_id, then build evaluation lists
# CHANGE: cells 4 and 5 consolidated into one cell.
# CHANGE: added retrieved_doc_ids extraction — required for Precision@K metrics.

gold_map = {item["query_id"]: item for item in benchmark}
pred_map = {item["query_id"]: item for item in outputs}

common_ids = [qid for qid in gold_map if qid in pred_map]
print(f"Matched {len(common_ids)} query IDs: {common_ids}")

predictions        = [pred_map[qid]["response"] for qid in common_ids]
references         = [gold_map[qid]["answer"]   for qid in common_ids]
retrieved_contexts = [
    [ctx["text"]   for ctx in pred_map[qid].get("retrieved_context", [])]
    for qid in common_ids
]
# NEW: extract doc_id list per query for retrieval precision/recall
retrieved_doc_ids  = [
    [ctx["doc_id"] for ctx in pred_map[qid].get("retrieved_context", [])]
    for qid in common_ids
]

print("Example prediction:\n", predictions[0])
print("\nExample reference:\n", references[0])

In [ ]:
# Metric functions
# CHANGE: ROUGE, BLEU and faithfulness functions consolidated into one cell.
# CHANGE: bleu_score argument order fixed — original passed (ref_tokens, pred_tokens);
#         correct call is sentence_bleu(references, hypothesis).
# NEW: retrieval_precision_at_k and retrieval_recall_at_k added.

rouge  = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method1


def rouge_scores(pred, ref):
    scores = rouge.score(ref, pred)
    return {
        "rouge1_f": scores["rouge1"].fmeasure,
        "rouge2_f": scores["rouge2"].fmeasure,
        "rougeL_f": scores["rougeL"].fmeasure,
    }


def bleu_score(pred, ref):
    return sentence_bleu([ref.split()], pred.split(), smoothing_function=smooth)


def faithfulness_score(prediction, context_texts):
    """Token-overlap faithfulness: fraction of answer tokens found in context."""
    context = " ".join(context_texts).lower()
    pred_tokens = prediction.lower().split()
    if not pred_tokens:
        return 0.0
    supported = sum(1 for tok in pred_tokens if tok in context)
    return supported / len(pred_tokens)


# ─────────────────────────────────────────────────────────
# NEW: Retrieval Precision@K and Recall@K
#
# Precision@K: did we retrieve the correct source doc in top-K?
# (Binary — 1 if answer_source appears in retrieved_doc_ids[:K])
#
# This directly measures retrieval quality independently of
# generation quality, which is important for the rubric marks
# on the "Retrieval and Ranking" component.
# ─────────────────────────────────────────────────────────

def retrieval_precision_at_k(retrieved_ids, relevant_id, k=5):
    """1.0 if relevant_id appears in top-k retrieved doc ids, else 0.0."""
    return 1.0 if relevant_id in retrieved_ids[:k] else 0.0


def retrieval_recall_at_k(retrieved_ids, relevant_id, k=5):
    """Same as precision here since there is one relevant doc per query."""
    return retrieval_precision_at_k(retrieved_ids, relevant_id, k)


print("Metric functions defined.")

In [ ]:
# Compute BERTScore (batch operation — run once)

P, R, F1 = bertscore_score(predictions, references, lang="en", verbose=True)
bertscore_f = F1.tolist()

In [ ]:
# Evaluate each item — all metrics
# CHANGE: added difficulty, question_type, precision_at_1, precision_at_5
#         fields to each result dict.
# CHANGE: results stored in a pandas DataFrame for easy groupby analysis.

per_item = []

for i, qid in enumerate(common_ids):
    pred       = predictions[i]
    ref        = references[i]
    ctxs       = retrieved_contexts[i]
    ret_ids    = retrieved_doc_ids[i]
    answer_src = gold_map[qid]["answer_source"]
    difficulty = gold_map[qid].get("difficulty", "unknown")
    q_type     = gold_map[qid].get("question_type", "unknown")

    rouge_dict = rouge_scores(pred, ref)
    bleu       = bleu_score(pred, ref)
    faith      = faithfulness_score(pred, ctxs)
    prec_at_1  = retrieval_precision_at_k(ret_ids, answer_src, k=1)
    prec_at_5  = retrieval_precision_at_k(ret_ids, answer_src, k=5)

    per_item.append({
        "query_id":       qid,
        "difficulty":     difficulty,
        "question_type":  q_type,
        "rouge1_f":       rouge_dict["rouge1_f"],
        "rouge2_f":       rouge_dict["rouge2_f"],
        "rougeL_f":       rouge_dict["rougeL_f"],
        "bleu":           bleu,
        "bertscore_f":    bertscore_f[i],
        "faithfulness":   faith,
        "precision_at_1": prec_at_1,
        "precision_at_5": prec_at_5,
    })

df = pd.DataFrame(per_item)
df

In [ ]:
# Aggregate metrics
# CHANGE: now includes precision_at_1 and precision_at_5;
#         computes means from DataFrame columns for consistency.

metric_cols = [
    "rouge1_f", "rouge2_f", "rougeL_f",
    "bleu", "bertscore_f", "faithfulness",
    "precision_at_1", "precision_at_5"
]

aggregate = {f"mean_{col}": float(np.mean(df[col])) for col in metric_cols}

print("\nOverall Evaluation Results:")
print("-" * 45)
for k, v in aggregate.items():
    print(f"{k:30s}: {v:.4f}")

In [ ]:
# NEW: Per-difficulty and per-question-type breakdown
#
# Shows where the system excels and where it struggles —
# useful for the oral presentation and written analysis sections.

print("\nBreakdown by difficulty:")
difficulty_df = df.groupby("difficulty")[metric_cols].mean().round(3)
print(difficulty_df.to_string())

print("\nBreakdown by question type:")
type_df = df.groupby("question_type")[metric_cols].mean().round(3)
print(type_df.to_string())

In [ ]:
# NEW: Baseline comparison table
#
# Replaces the prose-only markdown cell from the original.
# Baseline = dense retrieval only (no BM25, no reranker),
# original all-MiniLM-L6-v2 embeddings, 80-token generation limit.
#
# These scores are from the original notebook run.
# Update BASELINE_SCORES if you have fresh baseline numbers.

BASELINE_SCORES = {
    "mean_rouge1_f":       0.4047,
    "mean_rouge2_f":       0.1723,
    "mean_rougeL_f":       0.3311,
    "mean_bleu":           0.0583,
    "mean_bertscore_f":    0.9014,
    "mean_faithfulness":   0.7908,
    "mean_precision_at_1": None,   # not measured in baseline
    "mean_precision_at_5": None,
}

print("\n" + "=" * 65)
print(f"{'Metric':<30} {'Baseline':>10} {'Improved':>10} {'Delta':>10}")
print("=" * 65)

for k in aggregate:
    improved = aggregate[k]
    baseline = BASELINE_SCORES.get(k)
    if baseline is not None:
        delta = improved - baseline
        sign  = "+" if delta >= 0 else ""
        print(f"{k:<30} {baseline:>10.4f} {improved:>10.4f} {sign+f'{delta:.4f}':>10}")
    else:
        print(f"{k:<30} {'N/A':>10} {improved:>10.4f} {'NEW':>10}")

print("=" * 65)

In [ ]:
# NEW: Retrieval audit — did we fetch the right document for each query?
#
# Flags any query where the ground-truth source doc was not retrieved
# in the top-5, making it easy to identify retrieval failures.

print("\nRetrieval audit (did we fetch the right document?):")
print(f"{'Query ID':<10} {'P@1':>5} {'P@5':>5} {'Source doc':<30} {'Top retrieved doc'}")
print("-" * 80)

for i, qid in enumerate(common_ids):
    ret_ids    = retrieved_doc_ids[i]
    answer_src = gold_map[qid]["answer_source"]
    p1  = df.loc[df["query_id"] == qid, "precision_at_1"].values[0]
    p5  = df.loc[df["query_id"] == qid, "precision_at_5"].values[0]
    top = ret_ids[0] if ret_ids else "NONE"
    flag = "" if p5 == 1.0 else " <- MISSED"
    print(f"{qid:<10} {p1:>5.0f} {p5:>5.0f} {answer_src:<30} {top}{flag}")

In [ ]:
# Save evaluation results
# CHANGE: output JSON now includes a baseline_comparison section
#         with per-metric deltas, enabling automated comparison scripts.

eval_results = {
    "per_item":  per_item,
    "aggregate": aggregate,
    "baseline_comparison": {
        "baseline": BASELINE_SCORES,
        "improved": aggregate,
        "delta": {
            k: round(aggregate[k] - BASELINE_SCORES[k], 4)
            for k in aggregate
            if BASELINE_SCORES.get(k) is not None
        }
    }
}

with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)

print(f"Saved evaluation results to {eval_path}")

In [ ]:
# Show aggregate results clearly (unchanged)

print("Evaluation Results (aggregate):")
for k, v in aggregate.items():
    print(f"{k:30s}: {v:.4f}")